In [ ]:

import sys
import subprocess
import json
import shutil
import textwrap
from pathlib import Path
from datetime import datetime
from xml.sax.saxutils import escape

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
        Image,
    )
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "pandas", "numpy", "matplotlib", "seaborn", "reportlab", "pyarrow"
    ])
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
        Image,
    )

# ============================================================
# 1) PROJECT ROOT DISCOVERY
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
PREPARED_CANDIDATE_DIRS = [
    PROJECT_ROOT / "data" / "prepared",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "silver",
    PROJECT_ROOT / "data" / "curated",
    PROJECT_ROOT / "data" / "features",
]
REPORTS_DIR = PROJECT_ROOT / "reports"
VALIDATION_DIR = REPORTS_DIR / "validation"
LOGS_DIR = PROJECT_ROOT / "logs"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_FILE_NAME = "05 Data Preparation- DM4ML-Group51.pdf"
OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILE_NAME

TMP_DIR = PROJECT_ROOT / "__tmp_data_prep_pdf_assets"
TMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_ROOT: {RAW_ROOT}")
print(f"BRONZE_ROOT: {BRONZE_ROOT}")
print(f"VALIDATION_DIR: {VALIDATION_DIR}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

# ============================================================
# 2) HELPERS
# ============================================================
def safe_str(x):
    try:
        return str(x)
    except Exception:
        return ""

def rel_path(path):
    try:
        return safe_str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return safe_str(path)

def collect_files(base_dir, patterns):
    results = []
    if not base_dir.exists():
        return results
    for pattern in patterns:
        results.extend(base_dir.rglob(pattern))
    return sorted(set(p for p in results if p.is_file()))

def latest_file(base_dir, patterns):
    files = collect_files(base_dir, patterns)
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

def file_info(path):
    if not path or not path.exists():
        return None
    st = path.stat()
    return {
        "name": path.name,
        "relative_path": rel_path(path),
        "size_kb": round(st.st_size / 1024, 2),
        "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "suffix": path.suffix.lower(),
    }

def read_text_preview(path, max_lines=50, max_chars=5000):
    if not path or not path.exists():
        return "File not found."
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()[:max_lines]
        joined = "\n".join(lines)
        return joined[:max_chars]
    except Exception as e:
        return f"Could not preview file: {e}"

def read_notebook_preview(path, max_code_cells=4, max_chars=7000):
    if not path or not path.exists():
        return "Notebook not found."
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = json.load(f)
        blocks = []
        code_idx = 0
        for cell in nb.get("cells", []):
            if cell.get("cell_type") != "code":
                continue
            code_idx += 1
            source = cell.get("source", [])
            source = "".join(source) if isinstance(source, list) else str(source)
            source = source.strip()
            if source:
                blocks.append(f"# Code cell {code_idx}\n{source}")
            if len("\n\n".join(blocks)) >= max_chars or code_idx >= max_code_cells:
                break
        out = "\n\n".join(blocks).strip()
        return out[:max_chars] if out else "No code cells found."
    except Exception as e:
        return f"Could not parse notebook: {e}"

def read_code_preview(path, max_lines=120, max_chars=7000):
    if not path or not path.exists():
        return "File not found."
    if path.suffix.lower() == ".ipynb":
        return read_notebook_preview(path, max_code_cells=max(1, 4), max_chars=max_chars)
    return read_text_preview(path, max_lines=max_lines, max_chars=max_chars)

def build_tree_text(base_path, max_depth=6, max_items=300):
    if not base_path.exists():
        return f"{base_path.name}/ (not found)"
    lines = [f"{base_path.name}/"]
    count = 0

    def walk(path, prefix="", depth=0):
        nonlocal count
        if depth >= max_depth or count >= max_items:
            return
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for idx, item in enumerate(items):
            if count >= max_items:
                break
            connector = "└── " if idx == len(items) - 1 else "├── "
            lines.append(prefix + connector + item.name + ("/" if item.is_dir() else ""))
            count += 1
            if item.is_dir():
                extension = "    " if idx == len(items) - 1 else "│   "
                walk(item, prefix + extension, depth + 1)

    walk(base_path)
    if count >= max_items:
        lines.append("... output truncated ...")
    return "\n".join(lines)

def wrap_block_text(text, width=95):
    wrapped = []
    for line in str(text).splitlines():
        if not line.strip():
            wrapped.append("")
            continue
        pieces = textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False,
        )
        wrapped.extend(pieces if pieces else [""])
    return "\n".join(wrapped)

def make_hashable_value(v):
    if isinstance(v, np.ndarray):
        return tuple(make_hashable_value(x) for x in v.tolist())
    if isinstance(v, list):
        return tuple(make_hashable_value(x) for x in v)
    if isinstance(v, tuple):
        return tuple(make_hashable_value(x) for x in v)
    if isinstance(v, set):
        return tuple(sorted(make_hashable_value(x) for x in v))
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True, ensure_ascii=False, default=str)
    try:
        hash(v)
        return v
    except TypeError:
        return str(v)

def make_display_value(v):
    if isinstance(v, np.ndarray):
        return json.dumps(v.tolist(), ensure_ascii=False)
    if isinstance(v, (list, tuple, set)):
        try:
            return json.dumps(list(v), ensure_ascii=False)
        except Exception:
            return str(v)
    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True, default=str)
        except Exception:
            return str(v)
    if pd.isna(v) if not isinstance(v, (list, tuple, set, dict, np.ndarray)) else False:
        return ""
    return safe_str(v)

def safe_duplicate_count(df):
    if df is None or df.empty:
        return 0
    tmp = df.copy()
    for col in tmp.columns:
        tmp[col] = tmp[col].map(make_hashable_value)
    return int(tmp.duplicated().sum())

def wrap_path_for_pdf(value, max_chunk=32):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    separators = {"\\", "/", "_", "-", "="}
    parts = []
    token = ""

    for ch in text:
        token += ch
        if ch in separators:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines = []
    current = ""

    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                subparts = textwrap.wrap(
                    part,
                    width=max_chunk,
                    break_long_words=True,
                    break_on_hyphens=True
                )
                if subparts:
                    lines.extend(subparts[:-1])
                    current = subparts[-1]
                else:
                    current = part

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=36):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                chunks = textwrap.wrap(
                    word,
                    width=max_len,
                    break_long_words=True,
                    break_on_hyphens=True,
                )
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = word
            else:
                current = word

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def to_para(value, style, kind="general"):
    if kind == "path":
        return Paragraph(wrap_path_for_pdf(value), style)
    return Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, header_bg="#D9EAD3", path_cols=None, file_cols=None):
    path_cols = path_cols or []
    file_cols = file_cols or []
    converted = []

    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            style = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), style))
            else:
                if c in path_cols or c in file_cols:
                    row_cells.append(to_para(cell, style, kind="path"))
                else:
                    row_cells.append(to_para(cell, style, kind="general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(header_bg)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return table

def df_to_wrapped_table(df, max_rows=20, col_widths=None):
    if df is None or df.empty:
        data = [["No data available"]]
    else:
        preview = df.head(max_rows).copy()
        for col in preview.columns:
            preview[col] = preview[col].map(make_display_value)
        data = [list(preview.columns)] + preview.astype(str).values.tolist()
    return make_wrapped_table(data, col_widths=col_widths)

def find_data_prep_assets():
    patterns = [
        "*prepare*.py", "*prepare*.ipynb",
        "*preprocess*.py", "*preprocess*.ipynb",
        "*clean*.py", "*clean*.ipynb",
        "*eda*.py", "*eda*.ipynb",
        "*explor*.py", "*explor*.ipynb",
        "*transform*.py", "*transform*.ipynb",
        "*feature*.py", "*feature*.ipynb",
    ]
    matches = []
    for base in [PROJECT_ROOT, SRC_DIR]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))

    cleaned = []
    for p in sorted(set(matches)):
        p_str = safe_str(p).lower()
        if ".ipynb_checkpoints" in p_str:
            continue
        if "/venv/" in p_str or "\\venv\\" in p_str or "/.venv/" in p_str or "\\.venv\\" in p_str:
            continue
        if "/site-packages/" in p_str or "\\site-packages\\" in p_str:
            continue
        cleaned.append(p)
    return cleaned

def find_prepared_datasets():
    patterns = ["**/*.csv", "**/*.parquet", "**/*.json"]
    found = []
    for base in PREPARED_CANDIDATE_DIRS:
        if base.exists():
            found.extend(collect_files(base, patterns))
    return sorted(set(found))

def load_table_file(path, nrows=None):
    if not path or not path.exists():
        return None
    try:
        suffix = path.suffix.lower()
        if suffix == ".csv":
            return pd.read_csv(path, nrows=nrows)
        if suffix == ".parquet":
            df = pd.read_parquet(path)
            return df.head(nrows) if nrows else df
        if suffix == ".json":
            with open(path, "r", encoding="utf-8") as f:
                payload = json.load(f)
            if isinstance(payload, list):
                return pd.DataFrame(payload).head(nrows) if nrows else pd.DataFrame(payload)
            if isinstance(payload, dict):
                for v in payload.values():
                    if isinstance(v, list) and v and isinstance(v[0], dict):
                        df = pd.DataFrame(v)
                        return df.head(nrows) if nrows else df
                return pd.DataFrame([payload]).head(nrows) if nrows else pd.DataFrame([payload])
    except Exception as e:
        print(f"Could not load {path}: {e}")
        return None
    return None

def discover_events_file():
    patterns = [
        "**/events.csv",
        "**/*events*.csv",
        "**/*interaction*.csv",
        "**/*clickstream*.csv",
        "**/*transactions*.csv",
    ]
    raw_matches = []
    for pattern in patterns:
        raw_matches.extend(RAW_ROOT.rglob(pattern) if RAW_ROOT.exists() else [])
    return max(raw_matches, key=lambda p: p.stat().st_mtime) if raw_matches else None

def discover_products_file():
    patterns = [
        "**/products.parquet",
        "**/products.csv",
        "**/products_raw.json",
        "**/*product*.parquet",
        "**/*product*.csv",
        "**/*product*.json",
    ]
    matches = []
    for base in [BRONZE_ROOT, RAW_ROOT]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def prepare_events_dataframe(df):
    if df is None or df.empty:
        return df, []

    notes = []
    df = df.copy()

    col_map = {c.lower(): c for c in df.columns}
    rename_dict = {}

    if "visitorid" in col_map:
        rename_dict[col_map["visitorid"]] = "user_id"
    elif "userid" in col_map:
        rename_dict[col_map["userid"]] = "user_id"

    if "itemid" in col_map:
        rename_dict[col_map["itemid"]] = "item_id"
    elif "productid" in col_map:
        rename_dict[col_map["productid"]] = "item_id"

    if "timestamp" in col_map:
        rename_dict[col_map["timestamp"]] = "event_ts"

    if "event" in col_map:
        rename_dict[col_map["event"]] = "event_type"
    elif "eventtype" in col_map:
        rename_dict[col_map["eventtype"]] = "event_type"

    df = df.rename(columns=rename_dict)
    notes.append("Standardized common interaction columns where present.")

    if "user_id" in df.columns or "item_id" in df.columns:
        existing_subset = [c for c in ["user_id", "item_id"] if c in df.columns]
        before = len(df)
        df = df.dropna(subset=existing_subset)
        notes.append(f"Dropped rows with missing user_id/item_id where applicable: {before - len(df)} rows removed.")

    if "event_type" in df.columns:
        df["event_type"] = df["event_type"].astype(str).str.strip().str.lower()
        notes.append("Normalized event_type to lowercase string values.")

    if "event_ts" in df.columns:
        ts_num = pd.to_numeric(df["event_ts"], errors="coerce")
        if ts_num.notna().sum() > 0:
            median_val = ts_num.dropna().median()
            if median_val > 1e12:
                df["event_ts"] = pd.to_datetime(ts_num, unit="ms", errors="coerce")
                notes.append("Parsed event_ts from epoch milliseconds.")
            elif median_val > 1e9:
                df["event_ts"] = pd.to_datetime(ts_num, unit="s", errors="coerce")
                notes.append("Parsed event_ts from epoch seconds.")
            else:
                df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
                notes.append("Parsed event_ts as datetime.")
        else:
            df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
            notes.append("Parsed event_ts as datetime.")

        missing_ts = int(df["event_ts"].isna().sum())
        if missing_ts > 0:
            notes.append(f"Found {missing_ts} rows with invalid timestamps after parsing.")

        df["event_hour"] = df["event_ts"].dt.hour
        df["event_date"] = df["event_ts"].dt.date.astype("string")

        event_ts_num = df["event_ts"].view("int64")
        valid_mask = df["event_ts"].notna()
        if valid_mask.any():
            min_val = event_ts_num[valid_mask].min()
            max_val = event_ts_num[valid_mask].max()
            if max_val != min_val:
                df["event_ts_norm"] = np.where(
                    valid_mask,
                    (event_ts_num - min_val) / (max_val - min_val),
                    np.nan
                )
                notes.append("Normalized event timestamp to event_ts_norm.")
            else:
                df["event_ts_norm"] = np.where(valid_mask, 0.0, np.nan)

    dedupe_cols = [c for c in ["user_id", "item_id", "event_type", "event_ts"] if c in df.columns]
    if dedupe_cols:
        before = len(df)
        df = df.drop_duplicates(subset=dedupe_cols)
        notes.append(f"Removed duplicate interactions using keys {', '.join(dedupe_cols)}: {before - len(df)} rows removed.")

    notes.append(f"Interaction rows after preparation: {len(df)}.")
    return df, notes

def prepare_products_dataframe(df):
    if df is None or df.empty:
        return df, []

    notes = []
    df = df.copy()

    # convert array-like values early to avoid downstream hashing/display issues
    for col in df.columns:
        df[col] = df[col].map(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)

    col_map = {c.lower(): c for c in df.columns}
    rename_dict = {}

    if "id" in col_map and "product_id" not in df.columns:
        rename_dict[col_map["id"]] = "product_id"
    if "category" in col_map:
        rename_dict[col_map["category"]] = "category"
    if "price" in col_map:
        rename_dict[col_map["price"]] = "price"
    if "title" in col_map and "title" not in df.columns:
        rename_dict[col_map["title"]] = "title"

    df = df.rename(columns=rename_dict)
    notes.append("Standardized common product columns where present.")

    if "category" in df.columns:
        df["category"] = df["category"].astype("string").fillna("unknown").str.strip().str.lower()
        top_cats = pd.get_dummies(df["category"], prefix="cat", dtype=int)

        if top_cats.shape[1] > 25:
            top_names = df["category"].value_counts().head(10).index.tolist()
            df["category_grouped"] = df["category"].where(df["category"].isin(top_names), "other")
            top_cats = pd.get_dummies(df["category_grouped"], prefix="cat", dtype=int)

        df = pd.concat([df, top_cats], axis=1)
        notes.append("Encoded category using one-hot columns.")

    if "price" in df.columns:
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        price_missing_before = int(df["price"].isna().sum())
        median_price = df["price"].median()
        if pd.notna(median_price):
            df["price"] = df["price"].fillna(median_price)
            notes.append("Filled missing price values with median.")
        price_min = df["price"].min()
        price_max = df["price"].max()
        if pd.notna(price_min) and pd.notna(price_max) and price_max != price_min:
            df["price_norm"] = (df["price"] - price_min) / (price_max - price_min)
            notes.append("Normalized price using min-max scaling.")
        else:
            df["price_norm"] = 0.0
            notes.append("Price normalization defaulted to 0.0 because price values were constant or invalid.")
        notes.append(f"Missing/invalid price count before imputation: {price_missing_before}.")

    if "product_id" in df.columns:
        before = len(df)
        # use hash-safe dedupe on product_id only
        pid = df["product_id"].map(make_hashable_value)
        df = df.loc[~pid.duplicated()].copy()
        notes.append(f"Removed duplicate product_id rows: {before - len(df)}.")

    notes.append(f"Product rows after preparation: {len(df)}.")
    return df, notes

def save_dataframe(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        df.to_parquet(path, index=False)
    elif path.suffix.lower() == ".csv":
        df.to_csv(path, index=False)
    else:
        df.to_csv(path.with_suffix(".csv"), index=False)

def choose_prepared_output_path(events_df=None, products_df=None):
    target_dir = PREPARED_CANDIDATE_DIRS[0]
    target_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if events_df is not None and not events_df.empty:
        return target_dir / f"prepared_interactions_{timestamp}.parquet"
    if products_df is not None and not products_df.empty:
        return target_dir / f"prepared_products_{timestamp}.parquet"
    return target_dir / f"prepared_dataset_{timestamp}.csv"

def dataset_summary(df, dataset_name):
    if df is None or df.empty:
        return {
            "dataset": dataset_name,
            "rows": 0,
            "columns": 0,
            "missing_cells": 0,
            "duplicate_rows": 0,
        }
    return {
        "dataset": dataset_name,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": safe_duplicate_count(df),
    }

def build_interaction_matrix_sample(events_df, max_users=25, max_items=25):
    if events_df is None or events_df.empty:
        return None
    if not {"user_id", "item_id"}.issubset(set(events_df.columns)):
        return None

    tmp = events_df[["user_id", "item_id"]].dropna().copy()
    if tmp.empty:
        return None

    top_users = tmp["user_id"].value_counts().head(max_users).index
    top_items = tmp["item_id"].value_counts().head(max_items).index
    tmp = tmp[tmp["user_id"].isin(top_users) & tmp["item_id"].isin(top_items)]

    if tmp.empty:
        return None

    mat = pd.crosstab(tmp["user_id"], tmp["item_id"])
    mat = (mat > 0).astype(int)
    return mat

def calculate_sparsity(events_df):
    if events_df is None or events_df.empty:
        return None
    if not {"user_id", "item_id"}.issubset(set(events_df.columns)):
        return None

    pairs = events_df[["user_id", "item_id"]].dropna()
    if pairs.empty:
        return None

    n_users = pairs["user_id"].nunique()
    n_items = pairs["item_id"].nunique()
    observed = pairs.drop_duplicates().shape[0]
    possible = n_users * n_items if n_users and n_items else 0
    if possible == 0:
        return None

    density = observed / possible
    sparsity = 1 - density
    return {
        "users": int(n_users),
        "items": int(n_items),
        "observed_pairs": int(observed),
        "possible_pairs": int(possible),
        "density": round(float(density), 6),
        "sparsity": round(float(sparsity), 6),
    }

def save_histogram(series, title, xlabel, filename):
    path = TMP_DIR / filename
    plt.figure(figsize=(8, 4.5))
    sns.histplot(series.dropna(), bins=30, color="#86BC25", edgecolor="white")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path

def save_barplot(series, title, xlabel, ylabel, filename, top_n=15):
    path = TMP_DIR / filename
    top = series.astype(str).value_counts().head(top_n)
    plt.figure(figsize=(8, 4.8))
    sns.barplot(x=top.values, y=top.index.astype(str), color="#007CB0")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path

def save_heatmap(df, title, filename):
    path = TMP_DIR / filename
    plt.figure(figsize=(8, 5.2))
    sns.heatmap(df, cmap="YlGnBu", cbar=True)
    plt.title(title)
    plt.xlabel("Items")
    plt.ylabel("Users")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    return path

# ============================================================
# 3) STYLES
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="CustomTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=14,
)

meta_style = ParagraphStyle(
    name="MetaStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=13,
    spaceAfter=5,
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    alignment=TA_LEFT,
    fontSize=12,
    leading=15,
    spaceAfter=8,
)

sub_heading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    alignment=TA_LEFT,
    fontSize=10.4,
    leading=12.5,
    spaceAfter=6,
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["BodyText"],
    alignment=TA_JUSTIFY,
    fontSize=10.0,
    leading=14,
    spaceAfter=8,
)

bullet_style = ParagraphStyle(
    name="BulletStyle",
    parent=styles["BodyText"],
    alignment=TA_LEFT,
    fontSize=10.0,
    leading=14,
    leftIndent=14,
    firstLineIndent=-8,
    spaceAfter=4,
)

code_style = ParagraphStyle(
    name="CodeStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=7.0,
    leading=8.4,
)

table_header_style = ParagraphStyle(
    name="TableHeaderStyle",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=8.1,
    leading=9.3,
    alignment=TA_LEFT,
)

table_cell_style = ParagraphStyle(
    name="TableCellStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.0,
    leading=8.5,
    alignment=TA_LEFT,
)

# ============================================================
# 4) DISCOVER PROJECT EVIDENCE
# ============================================================
data_prep_assets = find_data_prep_assets()
prepared_datasets_existing = find_prepared_datasets()

latest_validation_txt = latest_file(PROJECT_ROOT, ["**/run_validation.txt"])
latest_validation_json = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.json"])
latest_validation_pdf = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.pdf"])
latest_fix_log = latest_file(VALIDATION_DIR, ["**/fix_log_*.csv"])

events_file = discover_events_file()
products_file = discover_products_file()

events_raw_df = load_table_file(events_file)
products_raw_df = load_table_file(products_file)

events_prepared_df, events_notes = prepare_events_dataframe(events_raw_df)
products_prepared_df, products_notes = prepare_products_dataframe(products_raw_df)

prepared_output_path = None
if events_prepared_df is not None and not events_prepared_df.empty:
    prepared_output_path = choose_prepared_output_path(events_df=events_prepared_df)
    save_dataframe(events_prepared_df, prepared_output_path)
elif products_prepared_df is not None and not products_prepared_df.empty:
    prepared_output_path = choose_prepared_output_path(products_df=products_prepared_df)
    save_dataframe(products_prepared_df, prepared_output_path)

if prepared_output_path and prepared_output_path.exists():
    prepared_datasets_existing = sorted(set(prepared_datasets_existing + [prepared_output_path]))

events_summary = dataset_summary(events_prepared_df, "prepared_interactions")
products_summary = dataset_summary(products_prepared_df, "prepared_products")

sparsity_stats = calculate_sparsity(events_prepared_df)
interaction_matrix = build_interaction_matrix_sample(events_prepared_df)

validation_preview = read_text_preview(latest_validation_txt, max_lines=70, max_chars=7000)
validation_json_preview = read_text_preview(latest_validation_json, max_lines=70, max_chars=7000) if latest_validation_json else "No validation JSON report found."
fix_log_preview = read_text_preview(latest_fix_log, max_lines=60, max_chars=5000) if latest_fix_log else "No fix log found."

prep_asset_previews = []
for p in data_prep_assets[:5]:
    prep_asset_previews.append({
        "name": p.name,
        "relative_path": rel_path(p),
        "preview": read_code_preview(p, max_lines=120, max_chars=7000),
    })

raw_tree = build_tree_text(RAW_ROOT, max_depth=5, max_items=220)
prepared_tree_text = "\n\n".join(
    [build_tree_text(d, max_depth=5, max_items=160) for d in PREPARED_CANDIDATE_DIRS if d.exists()]
) if any(d.exists() for d in PREPARED_CANDIDATE_DIRS) else "No prepared/processed data folder found."

# ============================================================
# 5) GENERATE SUMMARY PLOTS
# ============================================================
plot_paths = []

if events_prepared_df is not None and not events_prepared_df.empty:
    if "event_type" in events_prepared_df.columns:
        plot_paths.append(save_barplot(
            events_prepared_df["event_type"],
            "Interaction Distribution by Event Type",
            "Count",
            "Event Type",
            "interaction_distribution.png",
            top_n=20
        ))

    if "item_id" in events_prepared_df.columns:
        plot_paths.append(save_barplot(
            events_prepared_df["item_id"],
            "Top Item Popularity",
            "Interactions",
            "Item ID",
            "item_popularity.png",
            top_n=15
        ))

    if "event_hour" in events_prepared_df.columns:
        plot_paths.append(save_histogram(
            events_prepared_df["event_hour"],
            "Interaction Hour Distribution",
            "Hour of Day",
            "interaction_hour_hist.png"
        ))

if products_prepared_df is not None and not products_prepared_df.empty and "price_norm" in products_prepared_df.columns:
    plot_paths.append(save_histogram(
        products_prepared_df["price_norm"],
        "Normalized Price Distribution",
        "Normalized Price",
        "normalized_price_hist.png"
    ))

if interaction_matrix is not None and not interaction_matrix.empty:
    plot_paths.append(save_heatmap(
        interaction_matrix,
        "User-Item Sparsity Heatmap (Sample)",
        "sparsity_heatmap.png"
    ))

# ============================================================
# 6) BUILD TABLE DATA
# ============================================================
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]

asset_rows = [["Notebook / Script", "Relative Path", "Last Modified", "Size"]]
for p in data_prep_assets[:20]:
    info = file_info(p)
    asset_rows.append([
        info["name"],
        info["relative_path"],
        info["modified"],
        f"{info['size_kb']} KB",
    ])
if len(asset_rows) == 1:
    asset_rows.append(["No preparation notebook/script found", "-", "-", "-"])

prepared_rows = [["Prepared File", "Relative Path", "Type", "Last Modified", "Size"]]
for p in prepared_datasets_existing[:20]:
    info = file_info(p)
    prepared_rows.append([
        info["name"],
        info["relative_path"],
        info["suffix"] or "N/A",
        info["modified"],
        f"{info['size_kb']} KB",
    ])
if len(prepared_rows) == 1:
    prepared_rows.append(["No prepared dataset found", "-", "-", "-", "-"])

artifact_rows = [["Artifact", "Relative Path", "Last Modified", "Size"]]
for p in [events_file, products_file, latest_validation_txt, latest_validation_json, latest_validation_pdf, latest_fix_log]:
    if p and p.exists():
        info = file_info(p)
        artifact_rows.append([
            info["name"],
            info["relative_path"],
            info["modified"],
            f"{info['size_kb']} KB",
        ])
if len(artifact_rows) == 1:
    artifact_rows.append(["No supporting artifacts found", "-", "-", "-"])

summary_df = pd.DataFrame([
    events_summary,
    products_summary,
])

sparsity_df = pd.DataFrame([sparsity_stats]) if sparsity_stats else pd.DataFrame({
    "note": ["Sparsity could not be calculated because user_id/item_id columns were unavailable."]
})

prepared_preview_df = None
if prepared_output_path and prepared_output_path.exists():
    prepared_preview_df = load_table_file(prepared_output_path, nrows=20)

team_table = make_wrapped_table(team_data, col_widths=[4.0 * inch, 2.0 * inch])

asset_table = make_wrapped_table(
    asset_rows,
    col_widths=[1.65 * inch, 3.25 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

prepared_table = make_wrapped_table(
    prepared_rows,
    col_widths=[1.55 * inch, 2.95 * inch, 0.55 * inch, 1.05 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

artifact_table = make_wrapped_table(
    artifact_rows,
    col_widths=[1.70 * inch, 3.20 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

summary_table = df_to_wrapped_table(summary_df, max_rows=10)
sparsity_table = df_to_wrapped_table(sparsity_df, max_rows=10)
prepared_preview_table = df_to_wrapped_table(prepared_preview_df, max_rows=20)

# ============================================================
# 7) BUILD PDF STORY
# ============================================================
story = []

story.append(Paragraph("05 Data Preparation", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
story.append(team_table)
story.append(Spacer(1, 14))

story.append(Paragraph("1. Objective", heading_style))
story.append(Paragraph(
    "This report documents the data preparation stage by capturing project assets, preprocessing evidence, exploratory analysis outputs, and the prepared dataset generated for downstream transformation and modeling.",
    body_style
))

story.append(Paragraph("2. Objective Coverage", heading_style))
story.append(Paragraph("• Handle missing user-item interactions and remove invalid or duplicate interaction rows.", bullet_style))
story.append(Paragraph("• Encode categorical attributes such as product category using one-hot columns where present.", bullet_style))
story.append(Paragraph("• Normalize numerical variables such as price and parse or standardize timestamps.", bullet_style))
story.append(Paragraph("• Produce EDA outputs for interaction distributions, item popularity, and sparsity patterns.", bullet_style))
story.append(Paragraph("• Save a prepared dataset ready for the next pipeline stage.", bullet_style))
story.append(Spacer(1, 10))

story.append(Paragraph("3. Data Preparation Assets", heading_style))
story.append(Paragraph(
    "The following notebooks or scripts were identified as likely data preparation, cleaning, EDA, transformation, or feature-related assets in the project.",
    body_style
))
story.append(asset_table)
story.append(Spacer(1, 12))

story.append(Paragraph("4. Supporting Project and Log Artifacts", heading_style))
story.append(Paragraph(
    "These supporting files were discovered from raw, bronze, and validation outputs. Long paths are wrapped using real line breaks at safe separators to avoid PDF cut-off issues.",
    body_style
))
story.append(artifact_table)
story.append(Spacer(1, 12))

story.append(Paragraph("5. Raw and Prepared Storage Structure", heading_style))
story.append(Paragraph("<b>Raw Data Tree</b>", meta_style))
story.append(Preformatted(wrap_block_text(raw_tree, width=92), code_style))
story.append(Spacer(1, 8))
story.append(Paragraph("<b>Prepared Data Tree</b>", meta_style))
story.append(Preformatted(wrap_block_text(prepared_tree_text, width=92), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("6. Preparation Actions Applied", heading_style))
all_notes = []
all_notes.extend(events_notes)
all_notes.extend(products_notes)
if not all_notes:
    all_notes = ["No preparation actions could be inferred because suitable source files were not found."]
for note in all_notes:
    story.append(Paragraph(f"• {escape(note)}", bullet_style))
story.append(Spacer(1, 10))

story.append(Paragraph("7. Prepared Dataset Inventory", heading_style))
story.append(prepared_table)
story.append(Spacer(1, 12))

story.append(Paragraph("8. Dataset Quality Summary After Preparation", heading_style))
story.append(summary_table)
story.append(Spacer(1, 12))

story.append(Paragraph("9. Sparsity Summary", heading_style))
story.append(sparsity_table)
story.append(Spacer(1, 12))

story.append(Paragraph("10. Prepared Dataset Preview", heading_style))
story.append(prepared_preview_table)
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("11. Summary Plots", heading_style))
if plot_paths:
    story.append(Paragraph(
        "The following plots summarize key EDA views required for the assignment, including histograms and a heatmap when enough interaction data is available.",
        body_style
    ))
    for idx, plot_path in enumerate(plot_paths, start=1):
        story.append(Paragraph(f"Plot {idx}: {escape(plot_path.stem.replace('_', ' ').title())}", sub_heading_style))
        story.append(Image(str(plot_path), width=6.8 * inch, height=3.8 * inch))
        story.append(Spacer(1, 10))
else:
    story.append(Paragraph(
        "No plots were generated because the required source columns were not found in the discovered datasets.",
        body_style
    ))
story.append(Spacer(1, 8))

story.append(PageBreak())

story.append(Paragraph("12. Data Preparation Code / Notebook Preview", heading_style))
if prep_asset_previews:
    for item in prep_asset_previews:
        story.append(Paragraph(f"Asset: {escape(item['name'])}", sub_heading_style))
        story.append(Paragraph(f"<b>Path:</b> {escape(item['relative_path'])}", meta_style))
        story.append(Preformatted(wrap_block_text(item["preview"], width=95), code_style))
        story.append(Spacer(1, 10))
else:
    story.append(Paragraph("No preparation notebook or script preview is available.", body_style))
story.append(Spacer(1, 8))

story.append(Paragraph("13. Validation / Fix Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("14. Data Quality JSON Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_json_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("15. Fix Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(fix_log_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("16. Conclusion", heading_style))
story.append(Paragraph(
    "This document captures the project evidence for the Data Preparation stage, including cleaning and preprocessing logic, EDA plots, prepared dataset output, and supporting logs needed for a submission-ready PDF deliverable.",
    body_style
))

# ============================================================
# 8) BUILD PDF
# ============================================================
def build_pdf(path):
    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=0.50 * inch,
        leftMargin=0.50 * inch,
        topMargin=0.55 * inch,
        bottomMargin=0.55 * inch,
    )
    doc.build(story)

try:
    build_pdf(OUTPUT_PATH)
    print(f"\nPDF created successfully: {OUTPUT_PATH}")
except PermissionError:
    alt_path = PROJECT_ROOT / f"05 Data Preparation- DM4ML-Group51-{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    build_pdf(alt_path)
    print("\nOriginal PDF is likely open or locked.")
    print(f"Saved alternate file instead: {alt_path}")
finally:
    try:
        shutil.rmtree(TMP_DIR, ignore_errors=True)
    except Exception:
        pass


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
RAW_ROOT: C:\Users\barath\recomart-pipeline\data\raw
BRONZE_ROOT: C:\Users\barath\recomart-pipeline\data\bronze
VALIDATION_DIR: C:\Users\barath\recomart-pipeline\reports\validation
OUTPUT_PATH: C:\Users\barath\recomart-pipeline\05 Data Preparation- DM4ML-Group51.pdf


C:\Users\barath\AppData\Local\Temp\ipykernel_46280\3625751073.py:547: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  event_ts_num = df["event_ts"].view("int64")



PDF created successfully: C:\Users\barath\recomart-pipeline\05 Data Preparation- DM4ML-Group51.pdf
